# Tin can fall detection

Fine-tune YOLOv8n on labeled cans


In [12]:
!pip install -q ultralytics opencv-python-headless tqdm pyyaml "lap>=0.5.12"


In [ ]:
import shutil
import zipfile
from pathlib import Path
from collections import defaultdict

import lap
import yaml
import numpy as np
import cv2
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO

Device: cuda


In [14]:
INPUT_VIDEO = "/content/YTDown_YouTube_Reimagined-Tin-Can-Alley-Game_Media_B_qb3QZCXYU_002_720p (online-video-cutter.com).mp4"
OUTPUT_VIDEO = "/content/output_annotated.mp4"

DATASET_DIR = Path("/content/dataset")
WEIGHTS_DIR = Path("/content/weights")

CONF = 0.20
IMG_SIZE = 640
EPOCHS = 50

ASPECT_STANDING = 1.4
ASPECT_FALLEN = 0.9
SUSTAIN_FRAMES = 3
MIN_STANDING_FRAMES = 5
BASELINE_WINDOW_SEC = 2.0
SUMMARY_SECS = 5.0

DEDUP_IOU = 0.25

DATASET_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_PT = WEIGHTS_DIR / "cans_v1" / "weights" / "best.pt"
DATA_YAML = DATASET_DIR / "data.yaml"

assert Path(INPUT_VIDEO).exists(), f"Video not found: {INPUT_VIDEO}"
print("Input video found:", INPUT_VIDEO)


Input video found: /content/YTDown_YouTube_Reimagined-Tin-Can-Alley-Game_Media_B_qb3QZCXYU_002_720p (online-video-cutter.com).mp4


## Dataset


In [15]:
ZIP_PATH = "/content/brand of can soda.v1i.yolov8.zip"
REMAKE_DATASET_DIR = True
REMAP_TO_SINGLE_CLASS = True

zip_file = Path(ZIP_PATH)
assert zip_file.exists(), f"ZIP not found: {ZIP_PATH}"

if REMAKE_DATASET_DIR and DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as zf:
    zf.extractall(DATASET_DIR)

hits = list(DATASET_DIR.rglob("data.yaml"))
assert hits, f"No data.yaml found under {DATASET_DIR} after unzip."
DATA_YAML = hits[0]
REAL_DATASET_DIR = DATA_YAML.parent
print("Using DATA_YAML:", DATA_YAML)

with open(DATA_YAML, "r") as f:
    dcfg = yaml.safe_load(f)

print("Original classes:", dcfg.get("names"))

if REMAP_TO_SINGLE_CLASS:
    remapped_total = 0
    for split in ("train", "val", "test"):
        rel = dcfg.get(split)
        if not rel:
            continue
        split_img_path = (REAL_DATASET_DIR / rel).resolve()
        labels_dir = split_img_path.parent / "labels"
        if not labels_dir.exists():
            labels_dir = REAL_DATASET_DIR / split / "labels"
        if not labels_dir.exists():
            print(f"  WARNING: no labels dir for split={split}")
            continue
        count = 0
        for lbl in labels_dir.glob("*.txt"):
            lines = []
            for ln in lbl.read_text().splitlines():
                ln = ln.strip()
                if not ln:
                    continue
                p = ln.split()
                if len(p) >= 5:
                    p[0] = "0"
                    lines.append(" ".join(p))
            lbl.write_text("\n".join(lines) + ("\n" if lines else ""))
            count += 1
        print(f"  {split}: remapped {count} label files")
        remapped_total += count
    dcfg["names"] = ["can"]
    dcfg["nc"] = 1
    with open(DATA_YAML, "w") as f:
        yaml.safe_dump(dcfg, f)
    print(f"Total remapped: {remapped_total}")

bad = []
for lbl in list(DATASET_DIR.rglob("*.txt"))[:200]:
    for ln in lbl.read_text().splitlines():
        ln = ln.strip()
        if ln and ln.split()[0] != "0":
            bad.append(lbl.name)
            break
if bad:
    print(f"WARNING: {len(bad)} files still have non-zero class IDs!")
else:
    print("Sample check: label class IDs are 0 only.")

print("\nClasses:", dcfg["names"])
for split in ("train", "val", "test"):
    print(f"  {split}: {dcfg.get(split)}")


Using DATA_YAML: /content/dataset/data.yaml
Original classes: ['7Up', 'coca-cola', 'mountain dew', 'pepsi', 'royal', 'sprite']
  train: remapped 664 label files
  test: remapped 83 label files
Total remapped: 747

Classes: ['can']
  train: ../train/images
  val: ../valid/images
  test: ../test/images


## Train YOLOv8n

Fine-tune `yolov8n.pt` on dataset.


In [16]:
FORCE_RETRAIN = True

if BEST_PT.exists() and not FORCE_RETRAIN:
    print("Weights found, skipping training:", BEST_PT)
else:
    if BEST_PT.exists():
        shutil.rmtree(BEST_PT.parent.parent, ignore_errors=True)
        print("Removed old weights, retraining...")
    model = YOLO("yolov8n.pt")
    model.train(
        data=str(DATA_YAML),
        epochs=50,
        imgsz=IMG_SIZE,
        batch=16,
        device=0 if DEVICE == "cuda" else DEVICE,
        project=str(WEIGHTS_DIR),
        name="cans_v1",
        exist_ok=True,
        patience=15,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.5,
        fliplr=0.5,
        mosaic=1.0,
        scale=0.6,
        translate=0.2,
        degrees=5.0,
    )
    print("Training done. Best weights:", BEST_PT)


Removed old (bad-label) weights, retraining...
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.02, hsv_s=0.8, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cans_v1, nbs=64, nms=False, opset=None, optimize=False, optimiz

In [17]:
model = YOLO(str(BEST_PT))
m = model.val(
    data=str(DATA_YAML),
    imgsz=IMG_SIZE,
    device=0 if DEVICE == "cuda" else DEVICE,
    verbose=False,
)
print(
    f"mAP50: {m.box.map50:.3f}   "
    f"mAP50-95: {m.box.map:.3f}   "
    f"P: {m.box.mp:.3f}   R: {m.box.mr:.3f}"
)


Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1085.1±100.5 MB/s, size: 38.5 KB)
val: Scanning /content/dataset/valid/labels.cache... 83 images, 0 backgrounds, 78 corrupt: 100% ━━━━━━━━━━━━ 83/83 29.0Mit/s 0.0s
val: /content/dataset/valid/images/1000_F_431153816_YCQLtLr3KgDgXfrIRBWwvwBdciVXfyZ3_jpg.rf.1bd11979a7719e41b0b89384c5da3f85.jpg: ignoring corrupt image/label: Label class 3 exceeds dataset class count 1. Possible class labels are 0-0
val: /content/dataset/valid/images/10_jpg.rf.4815bbbb20d10d907c14d47a8e77ecfb.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: /content/dataset/valid/images/10_jpg.rf.7cdc1d0a8ad507ebd3fe1c567ab8a70e.jpg: ignoring corrupt image/label: Label class 1 exceeds dataset class count 1. Possible class labels are 0-0
val: /conten

## Track cans

Run the detector each frame and keep IDs with ByteTrack.


In [55]:
cap = cv2.VideoCapture(INPUT_VIDEO)
FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION = N_FRAMES / FPS
cap.release()
print(f"{W}x{H}  {FPS:.1f} fps  {N_FRAMES} frames  {DURATION:.1f}s")

model = YOLO(str(BEST_PT))
history = defaultdict(list)
per_frame = defaultdict(list)

for f, r in enumerate(
    tqdm(
        model.track(
            INPUT_VIDEO,
            tracker="bytetrack.yaml",
            persist=True,
            stream=True,
            conf=CONF,
            imgsz=IMG_SIZE,
            device=0 if DEVICE == "cuda" else DEVICE,
            verbose=False,
        ),
        total=N_FRAMES,
        desc="tracking",
    )
):
    if r.boxes is None or r.boxes.id is None:
        continue
    for tid, (x1, y1, x2, y2) in zip(
        r.boxes.id.int().cpu().tolist(),
        r.boxes.xyxy.cpu().numpy(),
    ):
        history[tid].append((f, float(x1), float(y1), float(x2), float(y2)))
        per_frame[f].append((tid, float(x1), float(y1), float(x2), float(y2)))

print(f"Unique IDs tracked: {len(history)}")
print("Detections in baseline window (first 60 frames):")
for fnum in sorted(per_frame.keys()):
    if fnum > 60:
        break
    ids = [t for t, *_ in per_frame[fnum]]
    print(f"  frame {fnum}: {len(ids)} cans  IDs={ids}")


1280x720  30.0 fps  902 frames  30.1s


tracking:   0%|          | 0/902 [00:00<?, ?it/s]

Unique IDs tracked: 10
Detections in baseline window (first 60 frames):
  frame 0: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 1: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 2: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 3: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 4: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 5: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 6: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 7: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 8: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 9: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 10: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 11: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 12: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 13: 6 cans  IDs=[1, 2, 3, 4, 5, 6]
  frame 14: 7 cans  IDs=[1, 2, 3, 4, 5, 6, 15]
  frame 15: 8 cans  IDs=[1, 2, 3, 4, 5, 6, 15, 16]
  frame 16: 7 cans  IDs=[1, 2, 3, 4, 5, 6, 15]
  frame 17: 8 cans  IDs=[1, 2, 3, 4, 5, 6, 15, 16]
  frame 18: 7 cans  IDs=[1, 2, 3, 4, 5, 6, 16]
  frame 19: 7 cans  IDs=[1, 2, 3, 4, 5, 6, 15]
  frame 20: 7 cans  IDs=[1, 2, 3, 4, 5, 6, 15]

In [56]:
LEDGE_MAX_CY = H * 0.58  # cans on the shelf sit higher than floor/couch FP boxes

valid_history = {}
rejected_ids = []
for tid, recs in history.items():
    centers_y = [(y1 + y2) / 2 for (_, x1, y1, x2, y2) in recs]
    if np.median(centers_y) <= LEDGE_MAX_CY:
        valid_history[tid] = recs
    else:
        rejected_ids.append(tid)

print(f"Removed {len(rejected_ids)} IDs below ledge: {sorted(rejected_ids)}")
history = valid_history

per_frame = defaultdict(list)
for tid, recs in history.items():
    for (f, x1, y1, x2, y2) in recs:
        per_frame[f].append((tid, x1, y1, x2, y2))


Removed 3 false-positive IDs: [16, 70, 71]


## Fall detection

Standing cans look tall (`h/w` high); tipped cans look wide (`h/w` low). 


In [57]:
baseline_f = int(BASELINE_WINDOW_SEC * FPS)


def _median_bbox(recs, max_f):
    in_win = [(x1, y1, x2, y2) for (f, x1, y1, x2, y2) in recs if f <= max_f]
    if not in_win:
        return None
    return tuple(np.median(np.array(in_win), axis=0))


def _iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    if inter == 0:
        return 0.0
    aa = (a[2] - a[0]) * (a[3] - a[1])
    bb = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (aa + bb - inter + 1e-6)


candidate_ids = [
    tid for tid, recs in history.items() if any(f <= baseline_f for (f, *_) in recs)
]
candidate_ids.sort(
    key=lambda t: sum(1 for (f, *_) in history[t] if f <= baseline_f),
    reverse=True,
)

starting_ids_list, kept_bboxes, id_alias = [], [], {}
for tid in candidate_ids:
    bb = _median_bbox(history[tid], baseline_f)
    if bb is None:
        continue
    merged = next(
        (
            kid
            for kid, kbb in zip(starting_ids_list, kept_bboxes)
            if _iou(bb, kbb) >= DEDUP_IOU
        ),
        None,
    )
    if merged is None:
        starting_ids_list.append(tid)
        kept_bboxes.append(bb)
    else:
        id_alias[tid] = merged

starting_ids = set(starting_ids_list)
N_CANS = len(starting_ids)
print(f"Cans at start of video: {N_CANS}  IDs: {sorted(starting_ids)}")
if id_alias:
    print(f"Merged duplicate IDs: {id_alias}")

combined_history = {tid: list(history[tid]) for tid in starting_ids}
for dup, kept in id_alias.items():
    combined_history[kept].extend(history[dup])
for tid in combined_history:
    combined_history[tid].sort(key=lambda r: r[0])

fall_events = {}

for tid in sorted(starting_ids):
    recs = combined_history[tid]
    standing_n, run_start, run_len = 0, None, 0

    for (f, x1, y1, x2, y2) in recs:
        hw = (y2 - y1) / max(1.0, x2 - x1)

        if hw >= ASPECT_STANDING:
            standing_n += 1
            run_start = run_len = 0
            continue

        if hw <= ASPECT_FALLEN and standing_n >= MIN_STANDING_FRAMES:
            if run_start is None:
                run_start, run_len = f, 1
            else:
                run_len += 1
            if run_len >= SUSTAIN_FRAMES:
                fall_events[tid] = {
                    "fall_frame": run_start,
                    "fall_seconds": run_start / FPS,
                    "last_bbox": (x1, y1, x2, y2),
                }
                break
        else:
            run_start = run_len = 0

for tid in sorted(starting_ids):
    if tid in fall_events:
        continue
    recs = combined_history[tid]
    standing_frames = [
        f
        for (f, x1, y1, x2, y2) in recs
        if (y2 - y1) / max(1.0, x2 - x1) >= ASPECT_STANDING
    ]
    if len(standing_frames) < MIN_STANDING_FRAMES:
        continue
    last_seen = max(f for (f, *_) in recs)
    if last_seen < N_FRAMES * 0.90:
        disappear_sec = last_seen / FPS
        last_rec = recs[-1]
        fall_events[tid] = {
            "fall_frame": last_seen,
            "fall_seconds": disappear_sec,
            "last_bbox": (last_rec[1], last_rec[2], last_rec[3], last_rec[4]),
        }

print(f"\nFalls detected: {len(fall_events)} / {N_CANS}")
for i, (tid, ev) in enumerate(
    sorted(fall_events.items(), key=lambda kv: kv[1]["fall_frame"]), 1
):
    t = ev["fall_seconds"]
    print(
        f"  Can #{i}  fell at {int(t // 60):02d}:{t % 60:05.2f}  (frame {ev['fall_frame']})"
    )

still_standing = sorted(set(starting_ids) - set(fall_events.keys()))
if still_standing:
    print(f"\nStill standing IDs: {still_standing}")


Cans at start of video: 7  IDs: [1, 2, 3, 4, 5, 6, 15]

Falls detected: 6 / 7
  Can #1  →  fell at 00:11.97  (frame 359)
  Can #2  →  fell at 00:14.37  (frame 431)
  Can #3  →  fell at 00:16.70  (frame 501)
  Can #4  →  fell at 00:19.17  (frame 575)
  Can #5  →  fell at 00:21.40  (frame 642)
  Can #6  →  fell at 00:23.47  (frame 704)

Still standing IDs: [15]


In [58]:
starting_ids = {tid for tid in starting_ids if tid in fall_events}
N_CANS = len(starting_ids)
print(f"Confirmed real cans: {N_CANS}  IDs: {sorted(starting_ids)}")


Confirmed real cans: 6  IDs: [1, 2, 3, 4, 5, 6]


## Render video


In [59]:
def fmt(sec):
    m = int(sec // 60)
    return f"{m:02d}:{sec - 60 * m:05.2f}"


frame_bboxes = defaultdict(dict)
for tid, recs in history.items():
    kid = id_alias.get(tid, tid)
    if kid not in starting_ids:
        continue
    for (f, x1, y1, x2, y2) in recs:
        frame_bboxes[f][kid] = (x1, y1, x2, y2)

fall_order = {
    tid: i
    for i, (tid, _) in enumerate(
        sorted(fall_events.items(), key=lambda kv: kv[1]["fall_frame"]), 1
    )
}

cap = cv2.VideoCapture(INPUT_VIDEO)
writer = cv2.VideoWriter(
    OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H)
)
last_bbox = {}
summary_start = max(0, N_FRAMES - int(SUMMARY_SECS * FPS))

for f in tqdm(range(N_FRAMES), desc="rendering"):
    ok, frame = cap.read()
    if not ok:
        break
    dets = frame_bboxes.get(f, {})
    last_bbox.update(dets)

    for tid in starting_ids:
        if tid in fall_events and f >= fall_events[tid]["fall_frame"]:
            continue
        bbox = dets.get(tid) or last_bbox.get(tid)
        if bbox is None:
            continue
        x1, y1, x2, y2 = map(int, bbox)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (200, 200, 200), 2)
        cv2.putText(
            frame,
            "can",
            (x1 + 4, max(14, y1 - 6)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (200, 200, 200),
            1,
            cv2.LINE_AA,
        )

    for tid, ev in fall_events.items():
        if f < ev["fall_frame"]:
            continue
        bbox = dets.get(tid) or last_bbox.get(tid) or ev["last_bbox"]
        x1, y1, x2, y2 = map(int, bbox)
        cx = (x1 + x2) // 2
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
        pts = np.array(
            [
                (cx, max(0, y1 - 18)),
                (cx - 10, max(0, y1 - 4)),
                (cx + 10, max(0, y1 - 4)),
            ]
        )
        cv2.drawContours(frame, [pts], 0, (0, 0, 255), -1)
        label = f"#{fall_order[tid]} fell @ {fmt(ev['fall_seconds'])}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        lx, ly = x1, max(th + 4, y1 - 22)
        cv2.rectangle(frame, (lx, ly - th - 4), (lx + tw + 6, ly + 2), (0, 0, 255), -1)
        cv2.putText(
            frame,
            label,
            (lx + 3, ly - 2),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1,
            cv2.LINE_AA,
        )

    if f >= summary_start:
        sorted_falls = sorted(fall_events.items(), key=lambda kv: kv[1]["fall_frame"])
        line_h = 24
        bw = 400
        bh = 64 + line_h * max(1, len(sorted_falls))
        x0, y0 = (W - bw) // 2, max(10, H - bh - 20)
        overlay = frame.copy()
        cv2.rectangle(overlay, (x0, y0), (x0 + bw, y0 + bh), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.65, frame, 0.35, 0, frame)
        cv2.putText(
            frame,
            f"Total time : {fmt(DURATION)}",
            (x0 + 14, y0 + 26),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )
        cv2.putText(
            frame,
            f"Cans down  : {len(fall_events)} / {N_CANS}",
            (x0 + 14, y0 + 52),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )
        for i, (_, ev) in enumerate(sorted_falls, 1):
            cv2.putText(
                frame,
                f"  Can #{i}  fell @ {fmt(ev['fall_seconds'])}",
                (x0 + 14, y0 + 52 + line_h * i),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.52,
                (220, 220, 220),
                1,
                cv2.LINE_AA,
            )

    writer.write(frame)

cap.release()
writer.release()
print("Done:", OUTPUT_VIDEO)
print(f"\nCans down: {len(fall_events)} / {N_CANS}   Duration: {fmt(DURATION)}")
print("\nFall timestamps:")
for i, (_, ev) in enumerate(
    sorted(fall_events.items(), key=lambda kv: kv[1]["fall_frame"]), 1
):
    print(f"  Can #{i}  {fmt(ev['fall_seconds'])}")


rendering:   0%|          | 0/902 [00:00<?, ?it/s]

Done: /content/output_annotated.mp4

Cans down: 6 / 6   Duration: 00:30.07

Fall timestamps:
  Can #1  →  00:11.97
  Can #2  →  00:14.37
  Can #3  →  00:16.70
  Can #4  →  00:19.17
  Can #5  →  00:21.40
  Can #6  →  00:23.47


In [ ]:
from google.colab import files

files.download(OUTPUT_VIDEO)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>